# scOPE end-to-end example

This notebook walks through the complete two-phase scOPE workflow:
1. **Phase 1** — learn latent space from bulk RNA-seq and train mutation classifiers.
2. **Phase 2** — project scRNA-seq into the bulk latent space, infer per-cell mutation probabilities.


In [47]:
import os

import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt

from scope import BulkPipeline, SingleCellPipeline
from scope.io import load_mutation_labels
from scope.visualization import (
    compute_umap,
    plot_mutation_probabilities,
    plot_scree,
    plot_mutation_heatmap,
)
from scope.evaluation import evaluate_all


## 1. Load data

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
BEATAML_DIR = "/Users/ashforda/Desktop/Pathways + Omics/scOPE/scOPE_project_overhaul/scOPE/data/BeatAML"
VANGALEN_DIR = "/Users/ashforda/Desktop/Pathways + Omics/scOPE/scOPE_project_overhaul/scOPE/data/vanGalen_AML_scRNA"
RAW_DIR      = os.path.join(BEATAML_DIR, "publicly_available_and_raw_counts")

bulk_tsv  = os.path.join(BEATAML_DIR, "beataml_waves1to4_norm_transposed.tsv")
mut_path  = os.path.join(RAW_DIR,     "beataml_wes_wv1to4_mutations_dbgap.txt")
clin_path = os.path.join(RAW_DIR,     "beataml_wv1to4_clinical.xlsx")
sc_path   = os.path.join(VANGALEN_DIR, "vanGalen_anndata.h5ad")

# ── Load bulk ──────────────────────────────────────────────────────────────
df_bulk = pd.read_csv(bulk_tsv, sep="\t", index_col=0)
adata_bulk = ad.AnnData(
    X   = df_bulk.values,
    obs = pd.DataFrame(index=df_bulk.index),
    var = pd.DataFrame(index=df_bulk.columns),
)

# ── Remap bulk Ensembl IDs → gene symbols ─────────────────────────────────
import mygene
mg = mygene.MyGeneInfo()
result = mg.querymany(
    adata_bulk.var_names.tolist(),
    scopes='ensembl.gene', fields='symbol',
    species='human', as_dataframe=True
)
id2sym = result['symbol'].dropna().to_dict()
new_var_names = [id2sym.get(g, g) for g in adata_bulk.var_names]
adata_bulk.var_names = pd.Index(new_var_names)
adata_bulk = adata_bulk[:, ~adata_bulk.var_names.str.startswith("ENSG")].copy()
print(f"Bulk after gene remapping: {adata_bulk.n_obs} samples × {adata_bulk.n_vars} genes")

# ── Build WES → RNA-seq ID map via clinical file ───────────────────────────
clin = pd.read_excel(clin_path)
wes_to_rna = (
    clin[['dbgap_dnaseq_sample', 'dbgap_rnaseq_sample']]
    .dropna()
    .set_index('dbgap_dnaseq_sample')['dbgap_rnaseq_sample']
    .to_dict()
)

# ── Build binary mutation matrix ───────────────────────────────────────────
AML_GENES = ["FLT3", "NPM1", "DNMT3A", "IDH1", "IDH2", "TET2", "RUNX1", "TP53", "NRAS", "KRAS"]

mut_raw = pd.read_csv(mut_path, sep="\t")
mut_raw['rnaseq_id'] = mut_raw['dbgap_sample_id'].map(wes_to_rna)
mut_raw = mut_raw.dropna(subset=['rnaseq_id'])

mut_matrix = (
    mut_raw[['rnaseq_id', 'symbol']]
    .drop_duplicates()
    .assign(mutated=1)
    .pivot_table(index='rnaseq_id', columns='symbol', values='mutated', fill_value=0)
)
mut_matrix.columns.name = None
mut_matrix.index.name   = None
mutation_labels = mut_matrix[[g for g in AML_GENES if g in mut_matrix.columns]]

# ── Align bulk samples to those with mutation calls ────────────────────────
shared = adata_bulk.obs_names.intersection(mutation_labels.index)
print(f"\nBulk samples total     : {adata_bulk.n_obs}")
print(f"Samples with mut calls : {len(mutation_labels)}")
print(f"Shared (overlap)       : {len(shared)}")

adata_bulk      = adata_bulk[shared].copy()
mutation_labels = mutation_labels.loc[shared]

# ── Load single-cell ───────────────────────────────────────────────────────
adata_sc = ad.read_h5ad(sc_path)

# ── Sanity check ───────────────────────────────────────────────────────────
print(f"\nBulk : {adata_bulk.n_obs} samples × {adata_bulk.n_vars} genes")
print(f"SC   : {adata_sc.n_obs} cells   × {adata_sc.n_vars} genes")
print(f"Mutations — shape: {mutation_labels.shape}")
print(f"\nMutation frequencies:\n{mutation_labels.sum().sort_values(ascending=False)}")
mutation_labels.head()


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


## 2. Phase 1 — Bulk pipeline

In [ ]:
# ── Inspect NaNs ───────────────────────────────────────────────────────────
X = adata_bulk.X
print(f"NaN count : {np.isnan(X).sum()}")
print(f"Inf count : {np.isinf(X).sum()}")
print(f"Zero rows (zero-count samples): {(X.sum(axis=1) == 0).sum()}")
print(f"Zero cols (zero-count genes)  : {(X.sum(axis=0) == 0).sum()}")

# ── Filter out zero-count samples and genes ────────────────────────────────
sample_mask = np.array(X.sum(axis=1) > 0).flatten()
gene_mask   = np.array(X.sum(axis=0) > 0).flatten()

adata_bulk = adata_bulk[sample_mask, gene_mask].copy()
print(f"\nAfter filtering — Bulk: {adata_bulk.n_obs} samples × {adata_bulk.n_vars} genes")

# ── Re-align mutation labels after filtering ───────────────────────────────
shared = adata_bulk.obs_names.intersection(mutation_labels.index)
adata_bulk      = adata_bulk[shared].copy()
mutation_labels = mutation_labels.loc[shared]
print(f"Shared samples after filtering: {len(shared)}")


In [ ]:
X = adata_bulk.X

print(f"NaN count  : {np.isnan(X).sum()}")
print(f"Inf count  : {np.isinf(X).sum()}")
print(f"Min value  : {np.nanmin(X)}")
print(f"Max value  : {np.nanmax(X)}")
print(f"Neg values : {(X < 0).sum()}")

# Where are the NaNs?
nan_rows, nan_cols = np.where(np.isnan(X))
print(f"NaN in {len(np.unique(nan_rows))} samples, {len(np.unique(nan_cols))} genes")


In [ ]:
bulk_pipe = BulkPipeline(
    norm_method='none',
    log1p=False,
    center=True,
    scale=True,
    decomposition='svd',
    n_components=500,
    classifier='logistic',
    classifier_kwargs={'C': 0.1, 'class_weight': 'balanced'},
)
bulk_pipe.fit(adata_bulk, mutation_labels, cv=5)

print('CV results:')
if bulk_pipe.cv_results_ is not None:
    summary = (
        bulk_pipe.cv_results_
        .groupby('mutation')[['auroc', 'auprc', 'brier']]
        .agg(['mean', 'std'])
        .round(3)
    )
    print(summary)
    

In [ ]:
print(bulk_pipe.cv_results_.columns.tolist())
print(bulk_pipe.cv_results_)


In [ ]:
# Scree plot
scree = bulk_pipe.decomposer_.scree_data()
fig, ax = plot_scree(scree, max_components=500)
plt.show()


## 3. Phase 2 — Single-cell projection

In [ ]:
print("Bulk var_names:", adata_bulk.var_names[:10].tolist())
print("SC var_names  :", adata_sc.var_names[:10].tolist())


In [ ]:
# Preprocessed bulk reference for moment matching
adata_bulk_pp = bulk_pipe.preprocessor_.transform(adata_bulk)

sc_pipe = SingleCellPipeline(
    bulk_pipeline=bulk_pipe,
    alignment_method='z_score_bulk',
    sc_min_counts=200,
    sc_min_genes=200,
)
sc_pipe.fit(adata_bulk_pp, adata_sc)
adata_sc = sc_pipe.transform(adata_sc)

# Show inferred mutation probability columns
prob_cols = [c for c in adata_sc.obs.columns if c.startswith('mutation_prob_')]
print('Mutation probability columns:', prob_cols)
adata_sc.obs[prob_cols].describe()


## 4. UMAP visualisation

In [ ]:
adata_sc = compute_umap(adata_sc, obsm_key=bulk_pipe.obsm_key_, n_neighbors=15)

fig = plot_mutation_probabilities(
    adata_sc,
    obsm_key='X_umap',
    ncols=3,
)
fig.savefig('mutation_probability_umap.pdf', bbox_inches='tight')
plt.show()


## 5. Cluster-level summary

In [ ]:
# Assumes leiden clustering was run e.g. via scanpy
# import scanpy as sc
# sc.pp.neighbors(adata_sc, use_rep=bulk_pipe.obsm_key_)
# sc.tl.leiden(adata_sc)

if 'leiden' in adata_sc.obs.columns:
    fig, ax = plot_mutation_heatmap(adata_sc, cluster_key='leiden')
    plt.show()


## 6. Save model

In [ ]:
bulk_pipe.save('models/bulk_pipeline.pkl')
adata_sc.write_h5ad('results/sc_with_mutation_probs.h5ad')
